## Importanto o dataset

In [ ]:
import requests, zipfile, io
import os

In [ ]:
url = "https://github.com/roneysco/Fake.br-Corpus/archive/refs/heads/master.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall(".")  # cria a pasta Fake.br-Corpus-master/

In [ ]:
os.listdir(".")

# Organizando colunas

In [ ]:
import pandas as pd
import glob, os

def carregar_textos(pasta, label):
    registros = []
    for caminho in sorted(glob.glob(os.path.join(pasta, "*.txt"))):
        repWith = ""
        if (label == 0):
            repWith = "t"
        id_noticia = os.path.basename(caminho).replace(".txt", repWith)
        with open(caminho, encoding="utf-8") as f:
            texto = f.read()
        registros.append({"id": id_noticia, "texto": texto, "label": label})
    return pd.DataFrame(registros)

df_fake = carregar_textos("Fake.br-Corpus-master/full_texts/fake", label=1)
df_true = carregar_textos("Fake.br-Corpus-master/full_texts/true", label=0)
df_textos = pd.concat([df_fake, df_true], ignore_index=True)

colunas_meta = [
    "autor",
    "link",
    "categoria",
    "data_publicacao",
    "num_tokens",
    "num_palavras",
    "num_types",
    "num_links",
    "num_maiusculas",
    "num_verbos",
    "num_verbos_subj_imp",
    "num_substantivos",
    "num_adjetivos",
    "num_adverbios",
    "num_verbos_modais",
    "num_pron_1_2_sing",
    "num_pron_1_plural",
    "num_pronomes",
    "pausalidade",
    "num_caracteres",
    "tam_medio_sentenca",
    "tam_medio_palavra",
    "pct_erros_ortograficos",
    "emotividade",
    "diversidade"
]


def carregar_metadados(pasta, label):
    registros = []
    for caminho in sorted(glob.glob(os.path.join(pasta, "*.txt"))):
        repWith = ""
        if (label == 0):
          repWith = "t"
        id_noticia = os.path.basename(caminho).replace("-meta.txt", repWith)
        with open(caminho, encoding="utf-8") as f:
            valores = [l.strip() for l in f.readlines()]
        registro = dict(zip(colunas_meta, valores))
        registro["id"], registro["label"] = id_noticia, label
        registros.append(registro)
    return pd.DataFrame(registros)

df_meta = pd.concat([
    carregar_metadados("Fake.br-Corpus-master/full_texts/fake-meta-information", 1),
    carregar_metadados("Fake.br-Corpus-master/full_texts/true-meta-information", 0),
], ignore_index=True)

df = df_textos.merge(df_meta.drop(columns="label"), on="id")

# Extração de metadados

In [ ]:
#Extração de metadados dos textos
import spacy
from collections import Counter

npl = spacy.load("pt_core_news_sm")

def metaExtractionFromText(text, dependency = True):
    types_counter  = Counter()
    types_counter['quant_tokens'] = len(text)
    
    for sentenca in text.sents:
        for token in sentenca:
            morph_str = str(token.morph)
            if morph_str:
                types_counter[morph_str] += 1
            if dependency:
                types_counter[token.dep_] +=1
    return types_counter


In [ ]:

def metaExtractionFromDataset(dataframe):
    docs = list(npl.pipe(dataframe, batch_size=100, disable=["ner"]))
    acumulador = Counter()
    for doc in docs:
        acumulador.update(metaExtractionFromText(doc))
    return dict(acumulador)


In [ ]:
def taxa_extracao(metadados):
    chaves = list(metadados.keys())
    for dado in chaves[1:]:
        metadados[dado] = metadados[dado]/metadados['quant_tokens']
    return metadados

In [ ]:
def collectMetadado(metadados, information):
    return  {chave: valor for chave, valor in metadados.items() if chave == information}

## EDA dos metadados extraídos

In [ ]:
textos_fake = list(npl.pipe(df_fake['texto'], batch_size=50, disable=["ner"]))

In [ ]:
acumulador = Counter()
for texto in textos_fake:
    acumulador.update(metaExtractionFromText(texto))
metadado_fake =  dict(acumulador)

In [ ]:
textos_true = list(npl.pipe(df_true['texto'], batch_size=50, disable=["ner"]))

In [ ]:
acumulador = Counter()

for doc in docs:
    acumulador.update(metaExtractionFromText(doc))
metadado_true =  dict(acumulador)

In [ ]:
import matplotlib.pyplot as plt

df = pd.DataFrame({'Notícias verdadeiras': metadado_true, 'Notícias falsas': metadado_fake})
df = df.fillna(0)

ax = df.plot(
    kind='bar', 
    figsize=(8, 5),
    width= 1
)

plt.title('Comparação de metadados de notícias falsas e verdadeiras')
plt.xlabel('Metadado')
plt.ylabel('Quantidade')
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:

metadados_fake_morph = collectMetadado(taxa_extracao(metadado_fake), 'morph')

plt.pie(
    metadados_fake_morph.values(), 
    labels= metadados_fake_morph.values(), 
    autopct='%1.1f%%', 
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
plt.title('Porcentagem de cada morfologia no dataset completo de Fakenews')
plt.show()

In [ ]:

metadados_true_morph = collectMetadado(taxa_extracao(metadado_true), 'morph')

plt.pie(
    metadados_true_morph.values(), 
    labels= metadados_true_morph.values(),
    autopct='%1.1f%%', 
    startangle=140,
    
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)

plt.title('Porcentagem de cada morfologia no dataset completo de notícias verdadeiras')
plt.show()

# Normalização e truncamento do texto

Os textos falsos e verdadeiros do Fake.br vieram de sites diferentes, com marcas de formatação próprias 
(caracteres de controle do cp1252, `[...]`, `..`, travessão e aspas curvas só nas verdadeiras). 
Sem normalizar, o modelo aprende a fonte em vez do conteúdo. 
As falsas também são bem mais curtas (mediana de 157 palavras contra 918), então o corte em 100 palavras deixa as classes com tamanho parecido.

In [ ]:
import re

def normalizar(texto):
    texto = str(texto)
    # Caracteres de controle do cp1252 (travessão e aspas que vieram com encoding quebrado)
    texto = texto.translate(str.maketrans({"\x96": "-", "\x97": "-", "\x93": '"', "\x94": '"', "\x91": "'", "\x92": "'"}))
    texto = re.sub(r"[\x80-\x9f]", " ", texto)
    # Tipografia: travessão e aspas curvas viram os equivalentes simples
    texto = re.sub(r"[–—]", "-", texto)
    texto = re.sub(r"[“”«»]", '"', texto)
    texto = re.sub(r"[‘’]", "'", texto)
    # [...], ..., .. e … viram um ponto só
    texto = re.sub(r"\[\.*\]|\.{2,}|…", ".", texto)
    # Números viram 0 (datas e valores diferem entre as fontes)
    texto = re.sub(r"\d", "0", texto)
    return " ".join(texto.split())

def truncar(texto, n=100):
    return " ".join(str(texto).split()[:n])

df["texto_trunc"] = df["texto"].apply(normalizar).apply(truncar)

# Remoção de stop words

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
stop_words_pt = set(stopwords.words("portuguese"))

# Versão sem stop words, usada só na análise (heatmap/SHAP). O modelo mantém as stop words,
# porque elas carregam estilo de escrita e melhoram o F1
padrao_stop_words = re.compile(
    r"\b(?:" + "|".join(map(re.escape, sorted(stop_words_pt, key=len, reverse=True))) + r")\b",
    flags=re.IGNORECASE,
)

def remover_stop_words(texto):
    return " ".join(padrao_stop_words.sub(" ", str(texto)).split())

df["texto_limpo"] = df["texto_trunc"].apply(remover_stop_words)

# Tratamento dos metadados

## Conversão de string para numero

In [ ]:
colunas_numericas = colunas_meta[4:]
df[colunas_numericas] = df[colunas_numericas].apply(pd.to_numeric, errors="coerce")

## Conversão de números abslutos para relativo a cada 100 palavras

In [ ]:
num_pal = df["num_palavras"]

In [ ]:
df["taxa_maiusculas"] = df["num_maiusculas"] / num_pal * 100
df["taxa_verbos"] = df["num_verbos"] / num_pal * 100
df['taxa_verbos_subj_imp'] = df["num_verbos_subj_imp"] / num_pal * 100
df['taxa_adjetivos'] = df["num_adjetivos"] / num_pal * 100
df['taxa_adverbios'] = df["num_adverbios"] / num_pal * 100
df['taxa_verbos_modais'] = df["num_verbos_modais"] / num_pal * 100
df['taxa_pron_1_2_sing'] = df["num_pron_1_2_sing"] / num_pal * 100
df['taxa_pron_1_plural'] = df["num_pron_1_plural"] / num_pal * 100
df['taxa_pronomes'] = df["num_pronomes"] / num_pal * 100
df['taxa_substantivos'] = df["num_substantivos"] / num_pal * 100

In [ ]:
def mattr(tokens, window_size=50):

    n = len(tokens)
    if n == 0:
        return 0.0
    if n <= window_size:
        return len(set(tokens)) / n  # cai pro TTR simples

    ratios = [
        len(set(tokens[i:i + window_size])) / window_size
        for i in range(n - window_size + 1)
    ]
    return sum(ratios) / len(ratios)


def calcular_diversidade(texto_trunc, window_size=100):
    tokens = str(texto_trunc).split()
    return mattr(tokens, window_size=window_size)


df["diversidade_relativa"] = df["texto_trunc"].apply(calcular_diversidade)


In [ ]:
metadados = [
    "taxa_maiusculas",
    "taxa_verbos",
    "taxa_verbos_subj_imp",
    "taxa_substantivos",
    "taxa_verbos_modais",
    "taxa_pron_1_2_sing",
    "taxa_pron_1_plural",
    "taxa_pronomes",
    "pausalidade",
    "pct_erros_ortograficos",
    "tam_medio_sentenca",
    "tam_medio_palavra",
    "diversidade_relativa",
]

In [ ]:
df = df[["texto_trunc", "texto_limpo"] + metadados + ["label"]].copy()

# Preparação para o treinamento dos modelos

## Split de treino e teste

In [ ]:
from sklearn.model_selection import train_test_split

# Separação entre Dados e Labels
X = df[["texto_trunc", "texto_limpo"] + metadados]
y = df["label"]

# Divisão entre split de treino e split teste
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## Imports

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate

## Pipelines

In [ ]:
txt_word = ("txt_word", TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True, token_pattern=r"(?u)\b[^\d\W]{2,}\b"), "texto_trunc")
txt_char = ("txt_char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=3, sublinear_tf=True), "texto_trunc")
meta     = ("meta", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), metadados)

configs = {
    "word":            [txt_word],
    "word+char":       [txt_word, txt_char],
    "word+meta":       [txt_word, meta],
    "word+char+meta":  [txt_word, txt_char, meta],
}

In [ ]:
import joblib

In [ ]:
joblib.dump((X_tr, X_te, y_tr, y_te), './dados_preparados.pkl')
joblib.dump(configs, './transformers.pkl')

In [ ]:
raise KeyboardInterrupt("Execução interrompida pelo usuário.")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt

ct_word = ColumnTransformer([txt_word])
X_word_tfidf = ct_word.fit_transform(X_tr)

svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X_word_tfidf)

plt.figure(figsize=(8,6))
plt.scatter(X_2d[:,0], X_2d[:,1], c=y_tr, cmap="coolwarm", alpha=0.5, s=10)
plt.xlabel("Componente 1")
plt.ylabel("Componente 2")
plt.title("Projeção 2D do TF-IDF (word) — fake vs real")
plt.colorbar(label="Classe")
plt.show()